### churn modeling silver layer 

In [0]:
%sql
select * from customer_360.bronze.bronze_churn_benchmark

In [0]:
%sql
CREATE OR REPLACE TABLE customer_360.silver.silver_churn_benchmark AS

WITH quality_passed AS (
  SELECT *
  FROM customer_360.bronze.bronze_churn_benchmark
  WHERE customer_id        IS NOT NULL
    AND credit_score        BETWEEN 300 AND 850
    AND age                 BETWEEN 18  AND 100
    AND tenure               BETWEEN 0   AND 15
    AND balance              >= 0
    AND num_of_products      BETWEEN 1   AND 4
    AND has_cr_card           IN (0, 1)
    AND is_active_member      IN (0, 1)
    AND exited                IN (0, 1)
    AND geography             IN ('France', 'Germany', 'Spain')
    AND gender                IN ('Male', 'Female')
    AND estimated_salary      >= 0
)

SELECT
  row_number,
  customer_id,
  TRIM(surname)                                        AS surname,
  credit_score,
  TRIM(geography)                                      AS geography,
  TRIM(gender)                                          AS gender,
  age,
  tenure,
  ROUND(balance, 2)                                     AS balance,
  num_of_products,
  has_cr_card,
  is_active_member,
  ROUND(estimated_salary, 2)                            AS estimated_salary,
  exited,

  CASE
    WHEN credit_score >= 800 THEN 'Excellent'
    WHEN credit_score >= 740 THEN 'Very Good'
    WHEN credit_score >= 670 THEN 'Good'
    WHEN credit_score >= 580 THEN 'Fair'
    ELSE                          'Poor'
  END                                                    AS credit_score_band,


  CASE
    WHEN age < 30  THEN '18-29'
    WHEN age < 40  THEN '30-39'
    WHEN age < 50  THEN '40-49'
    WHEN age < 60  THEN '50-59'
    ELSE                '60+'
  END                                                    AS age_band,

  CASE
    WHEN balance = 0                THEN 'Zero Balance'
    WHEN balance < 50000             THEN 'Low'
    WHEN balance < 100000            THEN 'Medium'
    WHEN balance < 150000            THEN 'High'
    ELSE                                   'Very High'
  END                                                    AS balance_tier,


  CASE
    WHEN num_of_products = 1 THEN 'Single Product'
    WHEN num_of_products = 2 THEN 'Two Products'
    ELSE                            'Three Plus Products'
  END                                                    AS product_category,

  CASE WHEN has_cr_card        = 1 THEN TRUE ELSE FALSE END AS has_credit_card,
  CASE WHEN is_active_member    = 1 THEN TRUE ELSE FALSE END AS is_active,
  CASE WHEN exited               = 1 THEN TRUE ELSE FALSE END AS is_churned,
  CASE WHEN exited               = 1 THEN 'Churned' ELSE 'Retained' END AS churn_label,


  CASE
    WHEN estimated_salary < 50000   THEN 'Low Income'
    WHEN estimated_salary < 100000  THEN 'Mid Income'
    WHEN estimated_salary < 150000  THEN 'Upper Mid Income'
    ELSE                                  'High Income'
  END                                                    AS salary_band,

  source_file,
  ingestion_time,
  current_timestamp()                                    AS silver_processed_time

FROM quality_passed;

In [0]:
%sql

SELECT
  COUNT(*)                                        AS total_rows,
  COUNT(DISTINCT customer_id)                      AS unique_customers,
  SUM(CASE WHEN is_churned THEN 1 ELSE 0 END)      AS churned,
  ROUND(AVG(CASE WHEN is_churned THEN 1.0 ELSE 0 END) * 100, 2) AS churn_rate_pct
FROM customer_360.silver.silver_churn_benchmark;

In [0]:
%sql
SELECT
  geography,
  COUNT(*)                                         AS customers,
  SUM(CASE WHEN is_churned THEN 1 ELSE 0 END)      AS churned,
  ROUND(AVG(CASE WHEN is_churned THEN 1.0 ELSE 0 END) * 100, 2) AS churn_rate_pct
FROM customer_360.silver.silver_churn_benchmark
GROUP BY geography
ORDER BY churn_rate_pct DESC;